# Installing dependencies

In [16]:
!pip install langgraph langchain-openai langchain-community faiss-cpu
!pip install langchain-ollama langchain-huggingface sentence-transformers

Defaulting to user installation because normal site-packages is not writeable


Defaulting to user installation because normal site-packages is not writeable


# Imports

In [17]:
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings

# Setup

In [19]:
llm = ChatOllama(model="llama3.2", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

texts = [
    "LangGraph is a library for building stateful, multi-step LLM workflows as graphs.",
    "Nodes are Python functions that read and update the shared state.",
    "Conditional edges route the flow based on the current state.",
    "MemorySaver checkpointer stores state per thread_id for conversation memory.",
]
vectorstore = FAISS.from_texts(texts, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

MAX_RETRIEVAL_RETRIES = 2
MAX_LLM_RETRIES = 2

Could not reach the Hub ([Errno 11001] getaddrinfo failed). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# State

In [ ]:
class GraphState(TypedDict, total=False):
    question: str
    rewritten_query: str
    documents: List[str]
    answer: str
    retrieval_attempts: int
    llm_attempts: int
    error: str
    history: List[Dict[str, str]]

# Nodes

In [ ]:
def question_node(state: GraphState):
    q = state["question"].strip()
    return {"question": q, "rewritten_query": q,
            "retrieval_attempts": 0, "llm_attempts": 0, "error": ""}

def retriever_node(state: GraphState):
    try:
        docs = retriever.invoke(state["rewritten_query"])
        return {"documents": [d.page_content for d in docs],
                "retrieval_attempts": state["retrieval_attempts"] + 1}
    except Exception as e:
        return {"documents": [], "error": f"retriever_error: {e}",
                "retrieval_attempts": state["retrieval_attempts"] + 1}

def rewrite_query_node(state: GraphState):
    prompt = f"Rewrite this question to be clearer for document search:\n{state['question']}"
    try:
        new_q = llm.invoke(prompt).content
    except Exception:
        new_q = state["question"]
    return {"rewritten_query": new_q}

def llm_node(state: GraphState):
    context = "\n".join(state["documents"])
    history = "\n".join(f"Q: {h['q']}\nA: {h['a']}" for h in state.get("history", [])[-3:])
    prompt = (
        "Answer ONLY using the context. If not present, say you don't know.\n\n"
        f"History:\n{history}\n\nContext:\n{context}\n\nQuestion: {state['question']}"
    )
    try:
        resp = llm.invoke(prompt).content
        return {"answer": resp, "error": "", "llm_attempts": state["llm_attempts"] + 1}
    except Exception as e:
        return {"error": f"llm_error: {e}", "llm_attempts": state["llm_attempts"] + 1}

def fallback_node(state: GraphState):
    return {"answer": "Sorry, I couldn't find reliable information to answer this. "
                      "Please rephrase or try again later."}

def answer_node(state: GraphState):
    return {"answer": state["answer"].strip()}

def memory_node(state: GraphState):
    history = state.get("history", []) + [{"q": state["question"], "a": state["answer"]}]
    return {"history": history}

# Routers (conditional edges)

In [ ]:
def route_after_retrieval(state: GraphState):
    if state.get("documents"):
        return "llm"
    if state["retrieval_attempts"] <= MAX_RETRIEVAL_RETRIES:
        return "rewrite"
    return "fallback"

def route_after_llm(state: GraphState):
    if not state.get("error"):
        return "answer"
    if state["llm_attempts"] < MAX_LLM_RETRIES:
        return "retry"
    return "fallback"

# Build Graph

In [ ]:
g = StateGraph(GraphState)
g.add_node("question", question_node)
g.add_node("retriever", retriever_node)
g.add_node("rewrite", rewrite_query_node)
g.add_node("llm", llm_node)
g.add_node("fallback", fallback_node)
g.add_node("answer", answer_node)
g.add_node("memory", memory_node)

g.set_entry_point("question")
g.add_edge("question", "retriever")
g.add_conditional_edges("retriever", route_after_retrieval,
                        {"llm": "llm", "rewrite": "rewrite", "fallback": "fallback"})
g.add_edge("rewrite", "retriever")                       # loop
g.add_conditional_edges("llm", route_after_llm,
                        {"answer": "answer", "retry": "llm", "fallback": "fallback"})
g.add_edge("fallback", "answer")
g.add_edge("answer", "memory")
g.add_edge("memory", END)

app = g.compile(checkpointer=MemorySaver())

# Run

In [20]:
config = {"configurable": {"thread_id": "user-1"}}
out = app.invoke({"question": "What are nodes in LangGraph?"}, config)
print(out["answer"])

out = app.invoke({"question": "And what does MemorySaver do?"}, config) 
print(out["answer"])

Nodes in LangGraph are Python functions.
MemorySaver is a checkpointer that stores state per thread_id for conversation memory.
